In [ ]:
# TOPIC 1 - PROMPT ENGINEERING
# Same task asked three ways, so students see better prompts give better answers.

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.bfloat16, device_map="auto")

def ask(text, n=120):
    msg = [{"role": "user", "content": text}]
    p = tok.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
    ids = tok(p, return_tensors="pt").to(model.device)
    out = model.generate(**ids, max_new_tokens=n, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()

# Zero-shot: just ask
print("Zero-shot:", ask("Is this review positive or negative? 'Broke in two days.'", 10))

# Few-shot: show the pattern first
print("Few-shot :", ask("good->Positive  bad->Negative  'fast and reliable'->", 5))

# Chain-of-thought: ask it to think step by step (gives smarter answers)
print("Step-by-step:", ask("Solve step by step: a train goes 60km in 1.5h. Average speed?"))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Zero-shot: This review is negative. The phrase "broke
Few-shot : Positive
Step-by-step: To find the average speed of the train, we can use the formula for average speed:

\[ \text{Average Speed} = \frac{\text{Total Distance}}{\text{Total Time}} \]

Given:
- Total distance traveled = 60 km
- Total time taken = 1.5 hours

Now plug these values into the formula:

\[ \text{Average Speed} = \frac{60 \text{ km}}{1.5 \text{ h}} \]

Next, perform the division:

\[ \text{Average Speed} = \frac{6


In [ ]:
# TOPIC 2 - HUGGING FACE ON NVIDIA H200
# Load a model on the GPU and see how much memory it uses.

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

# Check the GPU (on the lab box this shows the H200 with ~141 GB)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.bfloat16, device_map="auto")

# How much GPU memory the model takes
if torch.cuda.is_available():
    print("Model uses about", round(torch.cuda.memory_allocated()/1e9, 2), "GB of GPU memory")

# Generate one reply
msg = [{"role": "user", "content": "Explain what a GPU does in one sentence."}]
p = tok.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
ids = tok(p, return_tensors="pt").to(model.device)
out = model.generate(**ids, max_new_tokens=60, pad_token_id=tok.eos_token_id)
print("Reply:", tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip())

# The H200's big 141 GB memory is what lets you run bigger models and serve
# many users at once without running out of space.


GPU: CPU


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Reply: A GPU (Graphics Processing Unit) is designed to perform complex mathematical and computational tasks quickly and efficiently for applications like video games, scientific simulations, and data analysis.


In [ ]:
!pip install -q transformers peft accelerate sentencepiece
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

# Search for adapter folder
adapter_path = None

for root, dirs, files in os.walk("/content"):
    if "adapter_config.json" in files:
        adapter_path = root
        break

if adapter_path is None:
    print("❌ No LoRA adapter found.")
    print("You must first train and save the adapter.")
else:
    print("✅ Adapter found at:", adapter_path)

    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    # Load base model
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto"
    )

    # Load LoRA adapter
    model = PeftModel.from_pretrained(
        base_model,
        adapter_path
    )

    prompt = "Explain what a GPU does in one sentence."

    inputs = tokenizer(prompt, return_tensors="pt")

    if torch.cuda.is_available():
        inputs = {k: v.to("cuda") for k, v in inputs.items()}

    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        pad_token_id=tokenizer.eos_token_id
    )

    print("\n🤖 Response:")
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))

❌ No LoRA adapter found.
You must first train and save the adapter.


In [ ]:
!pip uninstall -y trl transformers peft
!pip install -q trl==0.9.6 transformers==4.41.2 peft==0.11.1 datasets accelerate bitsandbytes sentencepiece
import trl
import transformers
import peft

print("TRL:", trl.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    quantization_config=bnb_config,
    device_map="auto"
)

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

dataset = load_dataset(
    "yahma/alpaca-cleaned",
    split="train[:200]"
)

dataset = dataset.map(
    lambda x: {
        "text": f"### Instruction:\n{x['instruction']}\n\n### Response:\n{x['output']}"
    }
)

sft_config = SFTConfig(
    output_dir="./out",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    learning_rate=2e-4,
    bf16=False, # Changed to False as bf16 is not supported on CPU
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=sft_config,
    dataset_text_field="text",
    max_seq_length=512,
)

trainer.train()

model.save_pretrained("/content/day7-adapter")
tokenizer.save_pretrained("/content/day7-adapter")

print("✅ Adapter saved to /content/day7-adapter")

Found existing installation: trl 0.9.6
Uninstalling trl-0.9.6:
  Successfully uninstalled trl-0.9.6
Found existing installation: transformers 4.41.2
Uninstalling transformers-4.41.2:
  Successfully uninstalled transformers-4.41.2
Found existing installation: peft 0.11.1
Uninstalling peft-0.11.1:
  Successfully uninstalled peft-0.11.1
TRL: 1.5.1
Transformers: 5.9.0
PEFT: 0.19.1


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/default/ops.py:223: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


ValueError: Your setup doesn't support bf16/gpu. You need to assign use_cpu if you want to train the model on CPU.

In [ ]:
# --------------------------------------------------
# Load and use the trained LoRA adapter
# --------------------------------------------------

import torch
from transformers import AutoTokenizer
from peft import AutoPeftModelForCausalLM

ADAPTER_DIR = "./day7-adapter"

# Load tokenizer saved with the adapter
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)

# Load PEFT model (base model + LoRA adapter)
model = AutoPeftModelForCausalLM.from_pretrained(
    ADAPTER_DIR,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

model.eval()

# --------------------------------------------------
# Test prompt
# --------------------------------------------------

prompt = """### Instruction:
Explain what a GPU does in one sentence.

### Response:
"""

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=80,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True,
)

print("Reply:")
print(response.strip())

# --------------------------------------------------
# Optional: merge LoRA into base model
# --------------------------------------------------

merged_model = model.merge_and_unload()

merged_model.save_pretrained("./merged-model")
tokenizer.save_pretrained("./merged-model")

print("Merged model saved to ./merged-model")

OSError: Repo id must use alphanumeric chars, '-', '_' or '.'. The name cannot start or end with '-' or '.' and the maximum length is 96: './day7-adapter'.